In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../src').resolve()))

In [ ]:
import download_data_mem
import processing
import utils
import pandas as pd
import numpy as np
import sys
import plot_radiosonde_ddu_map
import plot_radiosonde_linear 
import plot_radiosonde_skewt
import cluster_profile
import importlib
import matplotlib.pyplot as plt
from cluster_profile import ClusterProfiles
from datetime import date, datetime, timedelta
from pathlib import Path


importlib.reload(download_data_mem)
importlib.reload(processing)
importlib.reload(utils)
importlib.reload(plot_radiosonde_linear)
importlib.reload(plot_radiosonde_skewt)
importlib.reload(plot_radiosonde_ddu_map)
importlib.reload(cluster_profile)

In [ ]:
summer = download_data_mem.get_data(years=[2020,2021,2022,2023,2024,2025],months=[12,1,2],hours = [0]) #DJF / MAM / JJA / SON iwith 2 cons. year
#avant c'était juste 2023,2024,2025
#transition = download_data_mem.get_data(years=[2020],months=[3,4,5,9,10,11],hours = [0])



In [ ]:
print("Summer:",len(summer))

In [ ]:
summer_int,summer_stats = utils.clean_and_interpolate(
    summer,
    z_grid=np.arange(50, 5000, 25),
    stitch=True,
    max_start_alt=200,   # strict : profil doit démarrer sous 200 m
    min_coverage=0.85    # doit atteindre 85 % de 5000 m ≈ 4300 m
) #try to 5km-6km

print("Summer:",len(summer_int))

utils.plot_vertical_coverage(summer_int)

In [ ]:
# INIT
cp = ClusterProfiles(variables=['t' ,'td','t-td','u','v']) #variables only useful for build_X function, not for build_features
#voir d'autres variables : rh, rhi ,... ?

# BUILD MATRIX
X = cp.build_X(summer_int,T_anomaly=True) #T_anomaly=True, to use temperature anomaly instead of absolute temperature

# CLEANING
X_clean = cp.treat_nan(strategy="mean")

# NORMALISATION
X_norm = cp.normalize_shape(['t','td','t-td'])


In [ ]:
print("X_norm shape:", X_norm.shape)

In [ ]:
# PCA
cp.find_nb_of_components()

In [ ]:
#apply PCA
X_red= cp.apply_pca(n_components=2)

In [ ]:
# CHOIX DU NOMBRE DE CLUSTERS
cp.find_nb_of_clusters(X_red,max_k=15)

In [ ]:
# FIT FINAL
labels = cp.fit_kmeans(k=7)
#plt.hist(labels, bins=np.arange(-0.5, 4.5, 1), rwidth=0.8)
plt.figure(figsize=(8,6))

for i in range(labels.min(), labels.max()+1):
    plt.scatter(
        X_red[labels == i, 0],
        X_red[labels == i, 1],
        label=f'Cluster {i}'
    )

plt.legend(title='Clusters')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('K-means Clusters in PCA Space')
plt.show()


In [ ]:
z_grid = np.arange(50,5000,25)
cp.plot_cluster_minipages(summer_int, z_grid[1:],quantiles=(0.25,0.75),ylim=(0, 5000))

FAIRE DES STATISTIQUES POUR LES DIFFERENTS CLUSTER (INVERSION,INVERSION HEIGHT, WIND COMPONENTS, ETC...)    plus intéressante que celle du dessous

In [ ]:
cp.cluster_summary_full(summer_int)

In [ ]:
fig = cp.plot_date_distribution(summer_int,figsize=(12,6))

In [ ]:
# Exemple été austral
fig, summary = cp.plot_cluster_seasonal_evolution(
    summer_int, season_months=[12, 1, 2])

In [ ]:
cp.plot_dendrogram(show_cut_at_k=6)
cp.silhouette_hierarchical(max_k=15)

In [ ]:
labels = cp.fit_hierarchical(k=6)
cp.plot_cluster_minipages(summer_int, z_grid[1:],quantiles=(0.25,0.75),ylim=(0, 5000))
cp.cluster_summary_full(summer_int)

In [ ]:
fig = cp.plot_date_distribution(summer_int,figsize=(12,6))

In [ ]:
# Exemple été austral
fig, summary = cp.plot_cluster_seasonal_evolution(
    summer_int, season_months=[ 12, 1, 2])

In [ ]:
def compute_mean_wind_by_cluster(soundings, labels, z_max=1000):

    import numpy as np

    clusters = np.unique(labels)

    results = {}

    for k in clusters:

        u_all = []
        v_all = []

        for i, s in enumerate(soundings):

            if labels[i] != k:
                continue

            df = s['data'].copy()

            if not all(var in df.columns for var in ['ff', 'dd', 'altitude']):
                continue

            # reconstruction u/v
            ff = df['ff'].values
            dd = np.deg2rad(df['dd'].values)

            u = -ff * np.sin(dd)
            v = -ff * np.cos(dd)

            z = df['altitude'].values
            mask = z <= z_max

            if np.sum(mask) < 3:
                continue

            u_all.extend(u[mask])
            v_all.extend(v[mask])

        if len(u_all) == 0:
            results[k] = (np.nan, np.nan, np.nan)
            continue

        u_mean = np.nanmean(u_all)
        v_mean = np.nanmean(v_all)
        wind_speed = np.nanmean(np.sqrt(np.array(u_all)**2 + np.array(v_all)**2))

        results[k] = (u_mean, v_mean, wind_speed)

    return results

In [ ]:
wind_stats = compute_mean_wind_by_cluster(summer_int, labels)

for k, (u, v, ws) in wind_stats.items():
    print(f"\nCluster {k}")
    print(f"Mean u: {u:.2f} m/s")
    print(f"Mean v: {v:.2f} m/s")
    print(f"Mean wind speed: {ws:.2f} m/s")